# Data Cleaning and Dataset Splitting

This notebook performs the initial data preparation steps for the credit card fraud detection project.

## Objectives

The main goals of this notebook are:

* Clean and validate the raw dataset
* Prepare the data for machine learning experiments
* Split the dataset into training and testing sets

## Data Source

The raw dataset is located at:

../data/raw/creditcard.csv

Relative path considering the notebook and the dataset are in different folders in the project root.

## Processing Steps

The following operations are performed in this notebook:

1. Load the raw dataset
2. Validate dataset integrity and basic statistics
3. Perform data cleaning steps
4. Prepare the feature matrix and target variable
5. Split the dataset into **training** and **test** sets

The resulting datasets are saved to:

- ..data/cleaned/creditcard_cleaned.parquet
- ..data/splits/train.parquet
- ..data/splits/test.parquet

Relative paths considering the notebook and the datasets are in different folders in the project root.

## Notes

This notebook was used during the **research phase** of the project to prototype data preparation steps.

The final implementation of the data preparation workflow is included in the **production pipeline** located in the `src/datapipeline` package.


In [ ]:
import pandas as pd
from pathlib import Path
from typing import Tuple
from sklearn.model_selection import train_test_split


# Config

In [ ]:
target_column = "Class"
dataset_path = '../data/raw/creditcard.csv'  #It is necessary to specify the path to the raw dataset
cleaned_df_path = '../data/cleaned' # location to save the cleaned dataset
cleaned_df_file_name = 'creditcard_cleaned.parquet'
test_size = 0.3 # proportion of the dataset to include in the test split
random_state = 42 # Controls the shuffling applied to the data before applying the split
splits_df_path = '../data/splits' # location to save the splitted dataset
min_samples = 1000 # minimun number of samples the dataset must contain
num_classes = 2 # minimum number of target classes the dataset must contain
train_df_file_name = 'train.parquet'
test_df_file_name = 'test.part'

# Load Data

In [ ]:
dataset_path = Path(dataset_path).resolve()
dataset_path

In [ ]:
data = pd.read_csv(dataset_path)

In [ ]:
data.describe()

# Validate Data

In [ ]:
def validate_data(
        df: pd.DataFrame,
        target_column: str,
        min_samples: int = 1000,
        num_classes: int = 2,
) -> None:
    
    """
    Validates if the dataset is valid

    Args:
        df (pd.DataFrame): pandas dataframe after preliminary cleaning
        target_column (str): target column
        min_samples (int, optional): required min number of samples. Defaults to 1000.
        num_classes (int, optional): number of classes in the target. Defaults to 2.
        logger (logging.Logger, optional): Logger instance.
  
    Raises:
        ValueError: the number of samples is less than min_samples
        ValueError: the number of classes is not equal to num_classes
        ValueError: one target class has no samples
    """

    print('Validating Data')
    
    if df.shape[0] < min_samples:
        raise ValueError(f"Dataset must have at least {min_samples} samples")
    
    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found")

    class_count = df[target_column].value_counts(dropna=True)

    if len(class_count) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, found {len(class_count)}")
    
    if class_count.min() <= 0:
     raise ValueError("One target class has no samples")
    print('Dataset Validated')

In [ ]:
validate_data(
        df = data,
        target_column = target_column,
        min_samples = min_samples,
        num_classes =num_classes)


# Data Cleaning

In [ ]:
# Path to save the clened dataframe
cleaned_df_path = Path(cleaned_df_path).resolve()
cleaned_df_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def clean_data(df: pd.DataFrame, 
               target_column: str, 
) -> Tuple[pd.DataFrame, int]:

    """
    Perform preliminary data cleaning.

    Steps:
    - Remove duplicate rows
    - Remove rows without target
    - Sanity check on target values

    Args:
        df (pd.DataFrame): Input dataframe.
        target_column (str): Name of target column.
        logger (logging.Logger, optional): Logger instance.

    Returns:
        Tuple[pd.DataFrame, int]: Cleaned dataframe and number of rows removed.
    """

    initial_rows = df.shape[0]

    # Remove duplicates
    df = df.drop_duplicates()
    duplicated_rows_removed = initial_rows - df.shape[0]
    
    print(f'Removed {duplicated_rows_removed} duplicated rows')


    # Remove rows without target
    before = df.shape[0]
    df = df.dropna(subset=[target_column])
    removed_missing_target = before - df.shape[0]
    
    print(f'Removed {removed_missing_target} rows without target')

    # Sanity checks
    if (df[target_column] < 0).any():
        raise ValueError("Invalid target values detected")

    total_removed = duplicated_rows_removed + removed_missing_target
    print(f'Total rows removed: {total_removed}')

    return df, total_removed



In [ ]:
df_cleaned, _ = clean_data(data,
                       target_column)


In [ ]:
df_cleaned.to_parquet(cleaned_df_path / cleaned_df_file_name)

# Data Splitting

In [ ]:
#Path to save the training and testing dataframes
splits_df_path = Path(splits_df_path).resolve()
splits_df_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def split_data(df: pd.DataFrame,
               target_column: str,
               test_size: float,
               random_state: int,
               ) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Split a DataFrame into training and testing sets.

    Args:
        df (pd.DataFrame): The DataFrame to split.
        target_column (str): The name of the target column.
        test_size (float): The proportion of the dataset to include in the test split.
        random_state (int): The seed used by the random number generator.
        logger (logging.Logger | None, optional): The logger to use. Defaults to None.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: A tuple containing the training and testing DataFrames.
    """

    X = df.drop(target_column, axis=1)
    y = df[target_column]

    print('Spliting data...')

    if y.nunique() < 2:
        raise ValueError("Target column must have at least two classes for stratified split")

        
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                        test_size=test_size, 
                                                        random_state=random_state,
                                                        stratify=y)
    print(f'{df.shape[0]} samples split into {X_train.shape[0]} ' 
                    f'train samples and {X_test.shape[0]} test samples')
    print(f'{X_train.shape[0]/df.shape[0]*100:.2f}% of the '
                    f'dataset is used for training and {X_test.shape[0]/df.shape[0]*100:.2f}% for testing' )

    df_train = X_train
    df_test  = X_test
    df_train[target_column] = y_train
    df_test[target_column] = y_test

    return df_train, df_test

In [ ]:
X_train, X_test = split_data(df = df_cleaned, 
                            target_column = target_column,
                            test_size = test_size,
                            random_state = random_state)

In [ ]:
X_train.to_parquet(splits_df_path / train_df_file_name)
X_test.to_parquet(splits_df_path / test_df_file_name)